### Libraries



In [3]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel, PeftConfig
from datasets import load_dataset
from evaluate import load as load_metric
from huggingface_hub import login

### Login to huggingface

In [2]:
login()

### Testing

In [5]:
# === SETTINGS ===
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_repo_id  = "eduhuemar001/tinyllama-german-checkpoints-sentiment"

# Load tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id)

# Load LoRA adapter configuration and model
model = PeftModel.from_pretrained(base_model, lora_repo_id)

# Optional: merge LoRA weights if you want a standalone model
model = model.merge_and_unload()

# Create text generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
prompt = "Prompt: What color is the sky? Answer shortly: "
result = generator(prompt, max_new_tokens=30)[0]["generated_text"].strip()
print(result)

Prompt: What color is the sky? Answer shortly: 

- The sky is blue.

- The sky is green.

- The sky is purple.

- The sky


In [ ]:
bleu = load_metric("bleu")
#grammar_prompts = [
#    ("Was ist der Plural von 'Haus'?", "Häuser"),
#    ("Welcher Artikel gehört zu 'Auto'?", "das"),
#    ("Setze den Satz ins Perfekt: Ich gehe zur Schule.", "Ich bin zur Schule gegangen."),
#]

#grammar_prompts = [
#    ("What is the plural of 'house'?", "houses"),
#    ("Which article belongs to 'car'?", "the"),
#    ("Put the sentence into present perfect: I go to school.", "I have gone to school."),
#]

grammar_prompts = [
    ("Answer briefly: What color is the sky?", "Blue"),
    ("Answer briefly: What color is grass?", "Green")
]

gen_outputs = []
references = []

print("\n=== German Grammar & Vocabulary ===")
for prompt, expected in grammar_prompts:
    result = generator(prompt, max_new_tokens=30)[0]["generated_text"].strip()
    print(f"Prompt: {prompt}\nModel: {result}\nExpected: {expected}\n")
    gen_outputs.append(result)
    references.append([expected])

#print("BLEU Score:", bleu.compute(predictions=gen_outputs, references=references))


=== German Grammar & Vocabulary ===
Prompt: Answer briefly: What color is the sky?
Model: Answer briefly: What color is the sky?

Student: Blue.

Teacher: Great! Now, can you tell me what the temperature is today?

Student:
Expected: Blue

Prompt: Answer briefly: What color is grass?
Model: Answer briefly: What color is grass?
Answer: Green.

2. What is the capital of the United States?
Answer: Washington, D.C.

3
Expected: Green



In [ ]:
print("\n=== Machine Translation (DE→EN) ===")
dataset_mt = load_dataset("wmt14", "de-en", split="test[:5]")
mt_outputs = []
mt_refs = []

for row in dataset_mt:
    input_text = f"Übersetze folgenden Satz ins Englische: {row['translation']['de']}"
    out = generator(input_text, max_new_tokens=50)[0]["generated_text"].replace(input_text, "").strip()
    print(f"DE: {row['translation']['de']}\nModel: {out}\nGT: {row['translation']['en']}\n")
    mt_outputs.append(out)
    mt_refs.append([row["translation"]["en"]])

print("BLEU (Translation):", bleu.compute(predictions=mt_outputs, references=mt_refs))


=== Machine Translation (DE→EN) ===


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

DE: Gutach: Noch mehr Sicherheit für Fußgänger
Model: und Autoverkehr.

Die Bundesregierung hat die Anzahl der Straßenbahnfahrzeuge in Deutschland auf 1000 pro Stunde reduziert.

Die Bundesregierung hat die Anzahl der Straßenbahnfahrzeuge
GT: Gutach: Increased safety for pedestrians

DE: Sie stehen keine 100 Meter voneinander entfernt: Am Dienstag ist in Gutach die neue B 33-Fußgängerampel am Dorfparkplatz in Betrieb genommen worden - in Sichtweite der älteren Rathausampel.
Model: Die Ampel wurde 1972 von der Stadt Gutach gebaut und 1973 in Betrieb genommen. Sie ist 100 Meter lang und 1,50 Meter hoch. Die Amp
GT: They are not even 100 metres apart: On Tuesday, the new B 33 pedestrian lights in Dorfparkplatz in Gutach became operational - within view of the existing Town Hall traffic lights.

DE: Zwei Anlagen so nah beieinander: Absicht oder Schildbürgerstreich?
Model: Das Wort „Schildbürgerstreich“ ist ein Wortspiel, das sich auf die Schildbürgerschaft bezieht. Die Schildbürgerschaft i